# 01 · ETL — coleta e limpeza dos dados (Eleições 2026, 1º turno)

**Disciplina:** Métodos Estatísticos em Ciência de Dados · FAETERJ · **Atividade assistida:** web scraping das eleições 2026
**Cargos:** Presidente · Senador · Deputado Federal · Deputado Estadual

Este é o **1º de 3 notebooks** (rode na ordem):

| Notebook | O que faz | Lê | Grava |
|---|---|---|---|
| **01_ETL** (este) | coleta no TSE, limpa, valida | TSE → `data/raw/` | `data/processed/*_1t.csv` |
| 02_Analise | métodos da aula: mediana, IQR, moda, Spearman, z-score, PCA | `data/processed/` | `data/processed/` + `reports/` (textos) |
| 03_Visualizacao | um gráfico por cargo + PCA | `data/processed/` | `reports/figures/` |

Fluxo da aula: **1 · Coletar → 2 · Limpar → 3 · Analisar.** Aqui estão os passos 1 e 2.

> **Reprodutibilidade:** os JSONs ficam em cache em `data/raw/` (pasta não versionada). Com o cache presente, nada é baixado
> de novo; para recoletar, apague `data/raw/` (o código mantém pausa de 2 s entre chamadas).

## 0. Configuração

In [1]:
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

# --- Pastas do repositório: os notebooks ficam em notebooks/, o resto é irmão dessa pasta ---
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"              # JSONs originais do TSE (não versionado, ver .gitignore)
PROC = ROOT / "data" / "processed"       # CSVs gerados pelo ETL e pelas análises (versionado)
FIG = ROOT / "reports" / "figures"       # gráficos
REP = ROOT / "reports"                   # textos entregáveis
for d in (RAW, PROC, FIG, REP):
    d.mkdir(parents=True, exist_ok=True)

# --- Parâmetros da coleta ---
BASE = "https://resultados.tse.jus.br/oficial"
CICLO = "ele2026"
HEADERS = {"User-Agent": "aula-ciencia-de-dados (FAETERJ)"}  # identifica a requisição (boa prática)
PAUSA = 2          # segundos entre requisições ao servidor (boa prática: 1 a 2 s)
TIMEOUT = 30

UFS = ["ac", "al", "am", "ap", "ba", "ce", "df", "es", "go", "ma", "mg", "ms", "mt", "pa",
       "pb", "pe", "pi", "pr", "rj", "rn", "ro", "rr", "rs", "sc", "se", "sp", "to"]
assert len(UFS) == 27

## 1. Coletar

### Passo 0 e 1 — como a página entrega os dados e onde está o JSON

Como mostrou a aula, o HTML inicial dos portais de apuração (ex.: CNN Brasil) traz só o aviso
*"Carregando dados da apuração..."*: os números chegam depois, por JavaScript. Por isso `requests + BeautifulSoup`
não enxergam os resultados. Dos três caminhos apresentados, usamos o **terceiro e mais estável: a fonte primária
(TSE)**. Os portais apenas reproduzem os resultados do TSE, que os publica em arquivos JSON.

O mapa dos arquivos está no JSON de configuração da apuração: `/oficial/comum/config/ele-c.json`. Ele informa o código de
cada eleição (`cd_eleicao`). O padrão do nome dos arquivos de votação por cargo e UF é:

```
{BASE}/ele2026/{cd_eleicao}/dados/{uf}/{uf}-c{cargo:04d}-e{cd_eleicao:06d}-u.json
```

Os parâmetros da URL (UF, cargo, eleição) mostram como repetir a coleta para as 27 UFs.

In [2]:
# Boa prática: consultar o robots.txt antes de automatizar.
r = requests.get("https://resultados.tse.jus.br/robots.txt", headers=HEADERS, timeout=TIMEOUT)
print("robots.txt -> HTTP", r.status_code, "|", r.headers.get("content-type"))
print(r.text[:160].replace("\n", " "))
time.sleep(PAUSA)

robots.txt -> HTTP 503 | text/plain; charset=utf-8
<HTML><HEAD> <TITLE>Service Unavailable</TITLE> </HEAD><BODY> <H1>Service Unavailable - DNS failure</H1> The server is temporarily unable to service your reques


> **Observação sobre o `robots.txt`:** o servidor não devolve um `robots.txt` válido (a resposta é uma página de erro
> do CDN, não um arquivo de regras), então não há restrição formal publicada para ler. Mesmo assim, mantivemos as
> demais boas práticas: arquivos públicos oficiais, `User-Agent` identificado, pausa de 2 s entre chamadas,
> cache local e nenhum laço infinito (número de requisições fixo e conhecido: 1 + 29 + 27×3 = 111).

In [3]:
LOG = []   # registro de cada requisição: fonte e horário da coleta (boa prática: "cite e valide")


def get_json(url, nome_cache, tentativas=3):
    """Baixa um JSON (com cache em disco, pausa e novas tentativas em caso de 429/5xx)."""
    arq = RAW / nome_cache
    if arq.exists():                                   # cache: não incomoda o servidor de novo
        return json.loads(arq.read_text(encoding="utf-8"))
    for n in range(1, tentativas + 1):
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        LOG.append({"url": url, "http": resp.status_code, "bytes": len(resp.content),
                    "coletado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
        if resp.status_code in (429, 500, 502, 503, 504) and n < tentativas:
            time.sleep(PAUSA * 3 * n)                  # reduz o ritmo (recomendação para 403/429)
            continue
        resp.raise_for_status()                        # falha clara se der erro HTTP
        dados = resp.json()
        arq.write_text(json.dumps(dados, ensure_ascii=False), encoding="utf-8")
        time.sleep(PAUSA)                              # pausa entre requisições
        return dados


# 1) Arquivo de configuração: de onde sai cada código de eleição
cfg = get_json(f"{BASE}/comum/config/ele-c.json", "ele-c.json")
print("Config do TSE gerada em:", cfg["dg"], cfg["hg"])

# 2) Descobre o código da eleição de cada cargo no 1º turno de 2026 (sem "chutar" números)
COD_ELEICAO = {}
for pleito in cfg["pl"]:
    if pleito["c"] != CICLO:
        continue
    for ele in pleito["e"]:
        if ele["t"] != "1" or "Ordinária" not in ele["nm"]:
            continue
        for abr in ele["abr"]:
            for cp in abr["cp"]:
                COD_ELEICAO[int(cp["cd"])] = (int(ele["cd"]), cp["ds"])
COD_ELEICAO = dict(sorted(COD_ELEICAO.items()))
pd.DataFrame([{"cod_cargo": k, "cargo": v[1], "cod_eleicao": v[0]} for k, v in COD_ELEICAO.items()])

Config do TSE gerada em: 15/10/2026 09:35:49


,cod_cargo,cargo,cod_eleicao
0,1,Presidente,6257
1,3,Governador,6259
2,5,Senador,6259
3,6,Deputado Federal,6259
4,7,Deputado Estadual,6259
5,8,Deputado Distrital,6259
6,25,Conselheiro Distrital,6261


### Passo 3 — os quatro cargos do 1º turno

| Cargo | Tipo | Observações usadas na coleta |
|---|---|---|
| Presidente (cód. 1) | majoritário · 1 voto | **Brasil (BR)**, as **27 UFs** e o **exterior (ZZ)** — as UFs alimentam o PCA e permitem conferir BR = soma das UFs |
| Senador (cód. 5) | majoritário · **2 votos** por eleitor em 2026 | 27 UFs; cuidado ao calcular percentuais |
| Deputado Federal (cód. 6) | proporcional · um conjunto por UF | 27 UFs; distribuição muito assimétrica |
| Deputado Estadual (cód. 7) | proporcional · por UF | 27 UFs; **no DF o cargo é Deputado Distrital (cód. 8)** |

**Estratégia:** laço sobre as UFs, pausa entre chamadas e um DataFrame por cargo.

In [4]:
CARGOS = {   # nome interno -> (código do cargo, unidades a coletar, código alternativo por UF)
    "presidente":        {"cod": 1, "unidades": ["br"] + UFS + ["zz"], "alt": {}},
    "senador":           {"cod": 5, "unidades": UFS, "alt": {}},
    "deputado_federal":  {"cod": 6, "unidades": UFS, "alt": {}},
    "deputado_estadual": {"cod": 7, "unidades": UFS, "alt": {"df": 8}},   # DF: Deputado Distrital
}


def url_votacao(cod_cargo, uf):
    cod_ele = COD_ELEICAO[cod_cargo][0]
    return (f"{BASE}/{CICLO}/{cod_ele}/dados/{uf}/{uf}-c{cod_cargo:04d}-e{cod_ele:06d}-u.json",
            f"{CICLO}_{uf}_c{cod_cargo:04d}.json")


BRUTOS = {}   # (cargo, unidade) -> JSON bruto
for cargo, cfg_c in CARGOS.items():
    for uf in cfg_c["unidades"]:
        cod = cfg_c["alt"].get(uf, cfg_c["cod"])
        url, nome = url_votacao(cod, uf)
        BRUTOS[(cargo, uf)] = get_json(url, nome)
    print(f"{cargo:18s} {len(cfg_c['unidades']):2d} unidades coletadas")

print(f"\nRequisições feitas nesta execução: {len(LOG)} (0 = tudo veio do cache em data/raw/)")
if LOG:
    pd.DataFrame(LOG).to_csv(PROC / "log_coleta.csv", index=False)   # fonte e horário de cada requisição

presidente         29 unidades coletadas
senador            27 unidades coletadas
deputado_federal   27 unidades coletadas
deputado_estadual  27 unidades coletadas

Requisições feitas nesta execução: 0 (0 = tudo veio do cache em data/raw/)


In [5]:
# Fonte e horário: cada arquivo do TSE informa quando foi gerado e quanto da apuração estava totalizado
meta = pd.DataFrame([{
    "cargo": c, "uf": u.upper(),
    "tse_gerado_em": pd.to_datetime(f'{d["dg"]} {d["hg"]}', format="%d/%m/%Y %H:%M:%S"),
    "totalizacao_final": d.get("tf"),                     # "s" = totalização final concluída
    "secoes_totalizadas_pct": float(d["s"]["pstn"]),
    "votos_nominais_validos_tse": int(d["v"]["vnom"]),
    "votos_anulados_sub_judice_tse": int(d["v"].get("vansj", 0)),
} for (c, u), d in BRUTOS.items()])

print("Seções totalizadas (min/max %):", meta.secoes_totalizadas_pct.min(), "/", meta.secoes_totalizadas_pct.max())
print("Arquivos gerados pelo TSE entre:", meta.tse_gerado_em.min(), "e", meta.tse_gerado_em.max())

# Atenção: nem toda disputa tem a totalização final concluída no arquivo do TSE
pendentes = meta[meta["totalizacao_final"] != "s"][["cargo", "uf", "tse_gerado_em", "totalizacao_final"]]
print("\nDisputas SEM totalização final no arquivo do TSE:")
print(pendentes.to_string(index=False) if len(pendentes) else "nenhuma")
meta.to_csv(PROC / "metadados_tse.csv", index=False)
meta.head()

Seções totalizadas (min/max %): 100.0 / 100.0
Arquivos gerados pelo TSE entre: 2026-10-04 20:51:22 e 2026-10-15 19:12:00

Disputas SEM totalização final no arquivo do TSE:
            cargo uf       tse_gerado_em totalizacao_final
 deputado_federal RO 2026-10-13 16:26:47                 n
deputado_estadual RO 2026-10-13 16:26:48                 n


,cargo,uf,tse_gerado_em,totalizacao_final,secoes_totalizadas_pct,votos_nominais_validos_tse,votos_anulados_sub_judice_tse
0,presidente,BR,2026-10-05 12:51:47,s,100.0,119300788,0
1,presidente,AC,2026-10-05 12:51:40,s,100.0,469066,0
2,presidente,AL,2026-10-05 12:51:41,s,100.0,1818856,0
3,presidente,AM,2026-10-05 12:51:41,s,100.0,2170763,0
4,presidente,AP,2026-10-05 12:51:40,s,100.0,464858,0


> **Atenção — Rondônia (Deputado Federal e Estadual):** o arquivo do TSE marca a totalização final como não concluída
> (`tf = "n"`) e traz o campo de resultado (eleito/suplente/não eleito) vazio. Os **votos estão completos** (100 % das seções)
> e conferem com o total do TSE; o que falta no arquivo é só a situação dos candidatos. Na mesma época a imprensa já divulgava
> os eleitos de RO, então o motivo da pendência no arquivo não está claro (não confirmado). Mantivemos os dados exatamente como o TSE
> os publicou, sem completar com fontes externas.

## 2. Limpar e organizar com pandas

Estrutura do JSON: `carg → agr (partido/federação) → par (partido) → cand (candidato)`; os votos de cada candidato estão
em `vap` (votos apurados). Achatamos isso em uma linha por **cargo × UF × candidato**, com as colunas padronizadas da
aula (`cargo, uf, candidato, partido, votos`) mais campos de apoio.

In [6]:
def json_para_df(dados, cargo, uf):
    """Achata o JSON do TSE em uma linha por candidato."""
    carg = dados["carg"][0]
    linhas = []
    for agr in carg["agr"]:
        federacao = agr["com"] if agr.get("tp") == "f" else ""
        for par in agr["par"]:
            for cand in par["cand"]:
                linhas.append({
                    "cargo": cargo,
                    "uf": uf.upper(),
                    "candidato": cand["nmu"],             # nome de urna
                    "partido": par["sg"],
                    "votos": cand["vap"],                 # ainda como texto no JSON
                    "numero": cand["n"],
                    "nome_completo": cand["nm"],
                    "federacao": federacao,
                    "situacao_candidatura": cand["dvt"],  # "Válido" ou "Anulado sub judice"
                    "resultado": cand["st"],              # Eleito / Não eleito / Suplente ...
                    "sq_candidato": cand["sqcand"],       # identificador único do candidato
                    "pct_validos_tse": cand["pvapn"],     # % calculado pelo TSE (para conferência)
                    "cargo_tse": carg["nmn"],             # no DF aparece "Deputado Distrital"
                })
    return pd.DataFrame(linhas)


bruto = pd.concat([json_para_df(d, c, u) for (c, u), d in BRUTOS.items()], ignore_index=True)
print("Linhas brutas:", len(bruto))
bruto.head(3)

Linhas brutas: 18998


,cargo,uf,candidato,partido,votos,numero,nome_completo,federacao,situacao_candidatura,resultado,sq_candidato,pct_validos_tse,cargo_tse
0,presidente,BR,FLAVIO BOLSONARO,PL,56104503,22,FLAVIO NANTES BOLSONARO,,Válido,2º turno,280002551544,"47,027772356",Presidente
1,presidente,BR,RONALDO CAIADO,PSD,2605148,55,RONALDO RAMOS CAIADO,,Válido,Não eleito,280002551932,"2,183680463",Presidente
2,presidente,BR,LULA,PT,53879538,13,LUIZ INÁCIO LULA DA SILVA,,Válido,2º turno,280002542548,"45,162767911",Presidente


In [7]:
df = bruto.copy()

# --- 1) Tipos: votos como número (to_numeric) ---
df["votos"] = pd.to_numeric(df["votos"], errors="coerce")
df["pct_validos_tse"] = pd.to_numeric(df["pct_validos_tse"].str.replace(",", ".", regex=False), errors="coerce")

# --- 2) Textos: remove espaços sobrando e padroniza a caixa das siglas ---
for col in ["candidato", "partido", "nome_completo", "federacao", "situacao_candidatura", "resultado"]:
    df[col] = df[col].astype("string").str.strip()
df["uf"] = df["uf"].str.upper()

# Rondônia (Dep. Federal e Estadual): o TSE deixa "resultado" vazio. Rotulamos explicitamente em vez de deixar vazio/NaN.
sem_res = df["resultado"].eq("")
print(f"Candidatos sem resultado final no arquivo do TSE: {int(sem_res.sum())} ->",
      sorted(df.loc[sem_res, ["cargo", "uf"]].drop_duplicates().apply(" ".join, axis=1)))
df.loc[sem_res, "resultado"] = "sem totalização final"

# --- 3) Duplicatas: a chave de um candidato é (cargo, uf, sq_candidato) ---
n_antes = len(df)
df = df.drop_duplicates(subset=["cargo", "uf", "sq_candidato"]).reset_index(drop=True)
print(f"Duplicatas removidas: {n_antes - len(df)}")

# --- 4) Percentual sobre os votos válidos de candidatos da mesma disputa (cargo × UF) ---
valido = df["situacao_candidatura"].eq("Válido")          # "Anulado sub judice" não conta como válido
base = df[valido].groupby(["cargo", "uf"])["votos"].sum().rename("votos_validos_total").reset_index()
df = df.merge(base, on=["cargo", "uf"], how="left")
df["pct_validos"] = (df["votos"] / df["votos_validos_total"] * 100).where(valido)
df = df.drop(columns="votos_validos_total")

ORDEM = ["cargo", "uf", "candidato", "partido", "votos", "pct_validos", "numero", "nome_completo",
         "federacao", "situacao_candidatura", "resultado", "sq_candidato", "pct_validos_tse", "cargo_tse"]
df = df[ORDEM].sort_values(["cargo", "uf", "votos"], ascending=[True, True, False]).reset_index(drop=True)
df.dtypes

Candidatos sem resultado final no arquivo do TSE: 373 -> ['deputado_estadual RO', 'deputado_federal RO']
Duplicatas removidas: 0


cargo                       str
uf                          str
candidato                string
partido                  string
votos                     int64
pct_validos             float64
numero                      str
nome_completo            string
federacao                string
situacao_candidatura     string
resultado                string
sq_candidato                str
pct_validos_tse         float64
cargo_tse                   str
dtype: object

### Checagens de validação

Quatro conferências da aula: votos numéricos, colunas padronizadas, sem duplicatas e **soma dos votos igual ao total do TSE**.
O arquivo do TSE informa o total de votos nominais válidos (`vnom`); a soma dos candidatos com situação *Válido* deve
bater exatamente (votos *Anulado sub judice* ficam fora, como no TSE).

In [8]:
# 1) votos numéricos, sem nulos nem negativos
print("votos nulos (NaN):", int(df["votos"].isna().sum()), "| negativos:", int((df["votos"] < 0).sum()))

# 2) colunas padronizadas
print("colunas-chave presentes:", all(c in df.columns for c in ["cargo", "uf", "candidato", "partido", "votos"]))

# 3) sem duplicatas
print("duplicatas (cargo, uf, sq_candidato):", int(df.duplicated(["cargo", "uf", "sq_candidato"]).sum()))

# 4) soma dos votos confere com o total do TSE, para cada cargo × unidade
soma = (df[df["situacao_candidatura"].eq("Válido")].groupby(["cargo", "uf"])["votos"].sum()
        .rename("soma_csv").reset_index())
conf = meta.merge(soma, on=["cargo", "uf"])
conf["diferenca"] = conf["soma_csv"] - conf["votos_nominais_validos_tse"]
resumo_conf = conf.groupby("cargo").agg(unidades=("uf", "count"),
                                        com_diferenca=("diferenca", lambda s: int((s != 0).sum())),
                                        soma_csv=("soma_csv", "sum"),
                                        total_tse=("votos_nominais_validos_tse", "sum"))
assert (conf["diferenca"] == 0).all(), "A soma dos votos não confere com o total do TSE!"
resumo_conf

votos nulos (NaN): 0 | negativos: 0
colunas-chave presentes: True
duplicatas (cargo, uf, sq_candidato): 0


,unidades,com_diferenca,soma_csv,total_tse
cargo,,,,
deputado_estadual,27,0,105468464,105468464
deputado_federal,27,0,110246229,110246229
presidente,29,0,238601576,238601576
senador,27,0,207038240,207038240


In [9]:
# Presidente: o total do Brasil deve ser igual à soma das 27 UFs + exterior (ZZ)
pres = df[df["cargo"] == "presidente"]
br = pres[pres["uf"] == "BR"].set_index("candidato")["votos"]
ufs_mais_zz = pres[pres["uf"] != "BR"].groupby("candidato")["votos"].sum()
print("BR == soma(27 UFs + ZZ) para todos os candidatos:", bool((br == ufs_mais_zz.reindex(br.index)).all()))

# Percentual calculado aqui x percentual do TSE (presidente, sem votos anulados sub judice)
dif_pct = (pres["pct_validos"] - pres["pct_validos_tse"]).abs().max()
print(f"maior diferença entre pct_validos e o pct do TSE: {dif_pct:.4f} p.p.")

# Cobertura: 27 UFs por cargo (+ BR e ZZ no presidente)
print(df.groupby("cargo")["uf"].nunique().rename("unidades_distintas").to_frame().T.to_string())

BR == soma(27 UFs + ZZ) para todos os candidatos: True
maior diferença entre pct_validos e o pct do TSE: 0.0000 p.p.


cargo               deputado_estadual  deputado_federal  presidente  senador
unidades_distintas                 27                27          29       27


### Salvando os CSVs dos 4 cargos em `data/processed/`

In [10]:
ARQUIVOS = {"presidente": "presidente_1t.csv", "senador": "senador_1t.csv",
            "deputado_federal": "deputado_federal_1t.csv", "deputado_estadual": "deputado_estadual_1t.csv"}
for cargo, nome in ARQUIVOS.items():
    # utf-8-sig: o Excel abre corretamente os acentos
    df[df["cargo"] == cargo].to_csv(PROC / nome, index=False, encoding="utf-8-sig")

pd.DataFrame({"arquivo": list(ARQUIVOS.values()),
              "linhas": [int((df["cargo"] == c).sum()) for c in ARQUIVOS],
              "ufs/unidades": [df.loc[df["cargo"] == c, "uf"].nunique() for c in ARQUIVOS],
              "votos_validos": [int(df[(df["cargo"] == c) & df["situacao_candidatura"].eq("Válido")
                                       & (df["uf"] != "BR")]["votos"].sum()) for c in ARQUIVOS]})

,arquivo,linhas,ufs/unidades,votos_validos
0,presidente_1t.csv,348,29,119300788
1,senador_1t.csv,301,27,207038240
2,deputado_federal_1t.csv,7262,27,110246229
3,deputado_estadual_1t.csv,11087,27,105468464


**Próximo passo:** abra `02_Analise.ipynb`.